# SEC Filings Pipeline

Full pipeline from scraping 10-K filings through BigQuery AI extraction and property graph creation.

## Pipeline Steps
1. **Google Cloud authentication** – gcloud auth
2. **00.0 Scraper** – Download 10-K filings from SEC.gov
3. **00.1 Parser** – Parse SGML to Markdown
4. **01 Extract Sections** – Extract Item 1, 1A, 7 sections to JSONL
5. **Upload to GCS** – Sync to BigQuery load bucket
6. **03 Init Tables** – Create sections, insights, staging tables
7. **04 Extraction** – AI extraction (Gemini) → insights
8. **05 Create Graph** – Transform insights to graph_edges
9. **06 Prepare Property Graph** – Node and edge tables
10. **06.1 Add Action to Market** – market_action column
11. **07 Create Property Graph DDL** – SecGraph property graph

## 0. Colab Setup
If you are running this in Google Colab, you need to clone the repository and install requirements to access the custom scraper scripts and SQL files.

In [ ]:
import sys
import os
if 'google.colab' in sys.modules:
    if not os.path.exists('00.0_scraper.py'):
        !git clone https://github.com/Kineviz/fortune500.git
        %cd fortune500
    !pip install -r requirements.txt
    !playwright install chromium


## 1. Setup & Google Cloud Authentication

In [ ]:
# Run gcloud authentication - opens browser for interactive login
# Required for: gsutil (GCS), bq (BigQuery)
!gcloud auth application-default login

# Optional: set project and ensure default credentials
# !gcloud config set project YOUR_PROJECT_ID

In [ ]:
# Verify authentication & set project
# If needed: !gcloud config set project YOUR_PROJECT_ID
!gcloud config list

In [ ]:
# Ensure we're in the project root (where 00.0_scraper.py, SQL files, list.csv live)
import os

def find_project_root():
    cwd = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(cwd, "00.0_scraper.py")):
            return cwd
        parent = os.path.dirname(cwd)
        if parent == cwd:
            break
        cwd = parent
    return os.getcwd()

root = find_project_root()
os.chdir(root)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Configure these for your environment
import os

GCS_BUCKET = "gs://kineviz_fortune500_sec_filing"  # Your GCS bucket
BQ_LOCATION = "US"
SGML_DIR = "data/sgml"
MARKDOWN_DIR = "data/markdown"
JSON_DIR = "data/json"

# Pipeline options
SCRAPER_LIMIT = 5        # Companies from list.csv (use small number for testing)
SCRAPER_LAST_N_YEARS = 2  # Years of filings to fetch
SCRAPER_OUTPUT = SGML_DIR

## 2. Run Scraper (00.0_scraper.py)

In [ ]:
import asyncio
import sys

# Add project root to path
sys.path.insert(0, os.getcwd())

from importlib.util import spec_from_file_location, module_from_spec

# Load scraper module
spec = spec_from_file_location("scraper", "00.0_scraper.py")
scraper_mod = module_from_spec(spec)
spec.loader.exec_module(scraper_mod)

scraper = scraper_mod.SECScraper(
    limit=SCRAPER_LIMIT,
    last_n_years=SCRAPER_LAST_N_YEARS,
    workers=5,
    output_dir=SCRAPER_OUTPUT,
    dry_run=False,
)

asyncio.run(scraper.run())

## 3. Run Parser (00.1_parser.py)

In [ ]:
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

# Import parser functions
spec = spec_from_file_location("parser", "00.1_parser.py")
parser_mod = module_from_spec(spec)
spec.loader.exec_module(parser_mod)

import multiprocessing

input_base = os.path.abspath(SGML_DIR)
output_base = os.path.abspath(MARKDOWN_DIR)

filings = []
for root, dirs, files in os.walk(input_base):
    for file in files:
        if file == "full-submission.txt":
            filings.append(os.path.join(root, file))

print(f"Found {len(filings)} filings to parse.")

tasks = [(f, input_base, output_base) for f in filings]

with ProcessPoolExecutor(max_workers=multiprocessing.cpu_count()) as executor:
    futures = [executor.submit(parser_mod.process_filing, t) for t in tasks]
    for _ in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Parsing"):
        pass

print("Parsing complete.")

## 4. Extract Sections (01_extract_sections.py)

In [ ]:
from concurrent.futures import ProcessPoolExecutor

spec = spec_from_file_location("extract", "01_extract_sections.py")
extract_mod = module_from_spec(spec)
spec.loader.exec_module(extract_mod)

input_base = os.path.abspath(SGML_DIR)
output_base = os.path.abspath(JSON_DIR)

tasks = []
for root, dirs, files in os.walk(input_base):
    if "full-submission.txt" in files:
        parts = root.split(os.sep)
        if "10-K" in parts:
            idx = parts.index("10-K")
            curr_ticker = parts[idx - 1]
            curr_year = "unknown"
            if idx >= 2 and len(parts[idx - 2]) == 4 and parts[idx - 2].isdigit():
                curr_year = parts[idx - 2]
            filepath = os.path.join(root, "full-submission.txt")
            tasks.append((filepath, output_base, curr_ticker, curr_year))

print(f"Found {len(tasks)} filings to process.")

with ProcessPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(extract_mod.process_filing, t) for t in tasks]
    for _ in tqdm(futures, desc="Extracting sections"):
        pass

print("Extraction complete.")

## 5. Upload JSONL to GCS

In [ ]:
!gsutil -m rsync -r {JSON_DIR} {GCS_BUCKET}/json

## 6. BigQuery Pipeline

In [ ]:
import subprocess
import glob

SCHEMA = "filing_id:STRING,company:STRING,company_name:STRING,cik:STRING,sic:STRING,irs_number:STRING,state_of_inc:STRING,org_name:STRING,sec_file_number:STRING,film_number:STRING,business_street_1:STRING,business_street_2:STRING,business_city:STRING,business_state:STRING,business_zip:STRING,business_phone:STRING,mail_street_1:STRING,mail_street_2:STRING,mail_city:STRING,mail_state:STRING,mail_zip:STRING,filing_url:STRING,year:INTEGER,section_id:STRING,content:STRING"


def run_bq_query(sql_file: str):
    with open(sql_file) as f:
        sql = f.read()
    subprocess.run(
        ["bq", "query", "--use_legacy_sql=false", f"--location={BQ_LOCATION}"],
        input=sql.encode(),
        check=True,
    )


def run_bq_load(uris: str):
    subprocess.run(
        [
            "bq", "load",
            "--source_format=NEWLINE_DELIMITED_JSON",
            "--replace",
            f"--schema={SCHEMA}",
            "sec_filings.sections_staging",
            uris,
        ],
        check=True,
        capture_output=True,
    )


def process_batch(uris: str):
    run_bq_load(uris)
    run_bq_query("04_extraction.sql")
    subprocess.run(
        ["bq", "query", "--use_legacy_sql=false", f"--location={BQ_LOCATION}",
         "INSERT INTO sec_filings.sections SELECT * FROM sec_filings.sections_staging"],
        check=True,
    )

In [ ]:
# 6.0 Initialize tables
run_bq_query("03_init_tables.sql")
print("Tables initialized.")

In [ ]:
# 6.1 Batch load, extract, archive
company_dirs = sorted(glob.glob(os.path.join(JSON_DIR, "*")))
company_dirs = [d for d in company_dirs if os.path.isdir(d)]

gcs_prefix = f"{GCS_BUCKET}/json"
total = len(company_dirs)

for i, company_dir in enumerate(tqdm(company_dirs, desc="Processing companies")):
    sections = glob.glob(os.path.join(company_dir, "*", "sections.jsonl"))
    if not sections:
        continue
    uris = [f"{gcs_prefix}/" + os.path.relpath(p, JSON_DIR).replace(os.sep, "/") for p in sections]
    uris_str = ",".join(uris)
    process_batch(uris_str)

In [ ]:
# 6.2 Create graph edges
run_bq_query("05_create_graph.sql")
print("Graph edges created.")

In [ ]:
# 6.3 Prepare node/edge tables
run_bq_query("06_prepare_property_graph.sql")
print("Property graph tables created.")

In [ ]:
# 6.4 Add market_action to nodes_market
# Note: If market_action column already exists, ALTER TABLE will fail on re-run.
# Edit 06_1_add_action_to_market.sql to remove the ADD COLUMN line in that case.
run_bq_query("06_1_add_action_to_market.sql")
print("Market action column added.")

In [ ]:
# 6.5 Create SecGraph property graph
run_bq_query("07_create_property_graph_ddl.sql")
print("Pipeline complete.")